In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os, sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "AreaAverages"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
RunType = ("TRACER","MOIST","NSSL")
SimulationTime = ("2022-06-30","2022-07-03")
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset):
    #3d variable case
    if ("nVertLevels" in variableSubset.dims):
        output = np.zeros((ModelData.Ntime,ModelData.Nzc))
    #3d variable case
    elif ("nVertLevelsP1" in variableSubset.dims):
        output = np.zeros((ModelData.Ntime,ModelData.Nzf))
    else:  #2d variable case
        output = np.zeros((ModelData.Ntime,1))
    return output

def GetMean(variableSubset):
    variableMean = variableSubset.mean(dim=("latitude","longitude")).data
    return variableMean

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def RunCalculations(varNames):
    outputDictionary={}
    
    num_times = ModelData.Ntime
    for count, t in enumerate(tqdm(range(num_times), desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
            
        #Loading Data
        [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
            #Subsetting Data

            variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
    
            #Initializing Output
            if count == 0:
                output = InitiateMatrix(variableSubset)
                outputDictionary[varName] = output

            #Taking Mean
            variableMean = GetMean(variableSubset)
            outputDictionary[varName][t] = variableMean


    outputDictionary[varName] = output
    return outputDictionary

# Notes:
# (1) may need to subset land/water later

In [ ]:
####################################
#CALCULATING

In [ ]:
#running
varNames = ["u10", "v10", "q2",
            "hfx", "qfx", "lh",
            "rainnc+rainc", "refl10cm_1km"]
varNames += ["w", "theta", "qv", "qc", "qc+qi", "qr", "refl10cm"]

#loading back in 
try:
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, outputDirectory, fileName = f"outputDictionary.h5")
    outputDictionary = DataSaving_Class.LoadDictionaryFromH5(filePath)
except Exception as e:
    print(f"Error: {e}")
    
    print("Running Calculation")
    outputDictionary = RunCalculations(varNames) #takes about 10 minutes
    #saving output
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, outputDirectory, fileName = f"outputDictionary.h5")
    DataSaving_Class.SaveDictionaryToH5(outputDictionary, filePath)

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def GetVerticalCoord(dataSubset):
    pressure_profile = dataSubset['pressure'].mean(dim=("latitude","longitude")).data
    dp = pressure_profile[-1] - pressure_profile[-2]
    p_topface = pressure_profile[-1] + dp  # extrapolate linearly
    pressure_profile_face = np.append(pressure_profile, p_topface)
    return (pressure_profile/100,pressure_profile_face/100)

[dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t=0)
pressure_profiles = GetVerticalCoord(dataSubset)
time_strings = ModelData.timeStrings
time = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

In [ ]:
import matplotlib.pyplot as plt

def PlotSingle(axis, varName, time, pressure_profiles, plottype="TZ"):
    """
    Plot one variable on a given Matplotlib axis.
    """

    # === Units and multiplier ===
    units = ModelData.GetUnits_Specific(varName).replace(" ", r"\ ")
    if varName in ["qv", "qc", "qi", "qr", "q2", "qfx"]:
        multiplier = 1e3
        units = units.replace('kg', 'g', 1)
    else:
        multiplier = 1
    
    output = multiplier * outputDictionary[varName]

    pressure_profile = pressure_profiles[0] if output.shape[1] == pressure_profiles[0].shape[0] else pressure_profiles[1]

    
    # === Plot ===
    def lineplot(time,output,varName,units):
        axis.plot(time, output.squeeze(), color="k")
        axis.set_ylabel(f"{varName} " + fr"$({units})$")
        axis.set_xlabel("Time")
        axis.grid(True)    
    if output.ndim == 1 or output.shape[1] == 1:
        lineplot(time,output,varName,units) #line plot
    else:
        if plottype=="TZ":
            pcm = axis.contourf(time, pressure_profile, output.T, cmap="viridis") #contour plot
            plt.colorbar(pcm, ax=axis, orientation="vertical", 
                         label=f"{varName} " + fr"$({units})$")
            axis.set_ylabel("Pressure (hPa)")
            axis.set_xlabel("Time")
            axis.invert_yaxis()
        elif plottype=="T":
            #taking mean
            output = np.mean(output, axis=1)
            lineplot(time,output,varName,units) #line plot
    
    axis.set_title(varName)
    plt.gcf().autofmt_xdate()

In [ ]:
####################################
#PLOTTING

In [ ]:
#PLOTTING

def MakeCombinedPlot(varNames,plottype):
    
    # --- Determine grid size automatically ---
    n_vars = len(varNames)
    n_cols = 3
    n_rows = int(np.ceil(n_vars / n_cols))
    
    # --- Create figure and GridSpec layout ---
    fig = plt.figure(figsize=(6 * n_cols, 3.5 * n_rows))
    gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, wspace=0.4, hspace=0.6)
    
    # --- Loop over variables and plot ---
    for i, varName in enumerate(varNames):
        row, col = divmod(i, n_cols)
        ax = fig.add_subplot(gs[row, col])
        PlotSingle(ax, varName, time, pressure_profiles, plottype=plottype)
    
    # --- Hide any unused grid cells (if grid larger than var count) ---
    for i in range(len(varNames), n_rows * n_cols):
        row, col = divmod(i, n_cols)
        ax = fig.add_subplot(gs[row, col])
        ax.axis("off")
    
    
    # --- Layout ---
    plt.subplots_adjust(left=0.07, right=0.97, bottom=0.07, top=0.93,
                        wspace=0.4, hspace=0.6)

    # --- Title
    plt.suptitle(f"{ModelData.region}/{ModelData.case}/{ModelData.mpType}")
    return fig



In [ ]:
for plottype in ["TZ","T"]:
    fig = MakeCombinedPlot(varNames,plottype=plottype)
    outputFile = os.path.join(outputPlottingDirectory, f"CombinedPlot_{plottype}.pdf")
    fig.savefig(outputFile, dpi=300, bbox_inches="tight")
    print(f"Saved to {outputFile}")